## Experiment to identify memory capacity in the RNN hidden state

### Recipe: experiment 1
- generate completely random sequence
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

### Recipe: experiment 2
- generate a sequence with pattern (1,2,3,4,5, 1,2,3,4,5, 1,2,3,4,5....)
- predict the next token
- stop after training till the loss saturates and see whether hidden states contain past memory
- plot reconstruction accuracy as a function of how far in the past the model reconstructs

In [43]:
## Load my files ##
import sys
sys.path.append('..')
from utils import get_sequence

## Load standard files ##
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim.lr_scheduler as lr_scheduler
from torch import from_numpy as tnsr
from scipy.stats import bernoulli
import torch.nn as nn
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist as dist
from sklearn.metrics.pairwise import cosine_similarity
from scipy.signal import find_peaks
from scipy.spatial.distance import cosine
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import torch.nn.functional as F
import random 
import pickle 

In [44]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")  # works only with NVIDIA GPUs (not on Mac)
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [45]:
def get_patterned_sequence(total_samples, token_number=7):
    sequence = []
    for i in range(total_samples):
        sequence.append(chr((i % token_number) + ord('A')))
    return sequence

In [46]:
def get_random_sequence(total_samples, token_number=7):
    return np.vectorize(chr)(np.random.randint(token_number, size=total_samples) + ord('A'))

In [47]:
# tokens start from 1
class Dataset_converter(Dataset):
    def __init__(self, data, working_memory=1, short_term_memory=1):
        
        self.X = np.zeros((((len(data)-working_memory-short_term_memory)), short_term_memory))
        self.y = np.zeros((((len(data)-working_memory-short_term_memory)), 1))

        for ii in range(self.X.shape[0]):
            for jj in range(self.X.shape[1]):
                self.X[ii,jj] = \
                ord(data[ii+jj])-64
      
            self.y[ii] = \
                ord(data[ii+jj+1])-64

        self.X = tnsr(self.X).long()
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index]

    def __len__(self):
        return self.X.shape[0]

In [48]:
# tokens start from 1
class Dataset_reconstructer(Dataset):
    def __init__(self, hidden_states, data, past_recall=1, short_term_memory=1):
        
        self.X = np.array(hidden_states)
        self.y = np.vectorize(ord)(data) - 64
        self.short_term_memory = short_term_memory
        self.past_recall = past_recall

        if short_term_memory == 1:
            self.y = np.concatenate(
                    (np.zeros(past_recall, dtype=int), self.y[:-past_recall])
                )

        self.X = tnsr(self.X)
        self.y = tnsr(self.y).long()

    def __getitem__(self, index):
        return self.X[index], self.y[index+self.short_term_memory-self.past_recall-1]

    def __len__(self):
        return self.X.shape[0]

In [49]:
class RNNEncoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True, nonlinearity='tanh')
        self.linear = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, h=None):
        embedded = self.embedding(x)
        out, h = self.rnn(embedded, h)
        out = self.linear(out[:,-1,:])

        return out, h  

In [50]:
class classifier(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.linear1 = nn.Linear(hidden_size, vocab_size)
        # self.linear2 = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        out = self.linear1(x)
        # out = self.linear2(x)

        return out  

In [51]:

reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_random_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_random.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 2.0142, accuracy: 0.1480
Iter : 2001, loss: 1.6150, accuracy: 0.1490
Iter : 3001, loss: 1.9088, accuracy: 0.1290
Iter : 4001, loss: 1.9498, accuracy: 0.1410
Iter : 5001, loss: 2.1432, accuracy: 0.1410
Iter : 6001, loss: 1.8657, accuracy: 0.1330
Iter : 7001, loss: 2.2944, accuracy: 0.1510
Iter : 8001, loss: 1.8405, accuracy: 0.1350
Iter : 9001, loss: 1.8880, accuracy: 0.1330
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.31it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.32it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.8904671401420426
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 0.6305936530183202
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.30it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.42997297026729403
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.30it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.328461307438182


In [52]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_patterned_sequence(total_samples, token_number=vocab_size-1)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Totoal accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_patterned.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 0.0135, accuracy: 0.9710
Iter : 2001, loss: 0.0038, accuracy: 1.0000
Iter : 3001, loss: 0.0018, accuracy: 1.0000
Iter : 4001, loss: 0.0008, accuracy: 1.0000
Iter : 5001, loss: 0.0004, accuracy: 1.0000
Iter : 6001, loss: 0.0002, accuracy: 1.0000
Iter : 7001, loss: 0.0001, accuracy: 1.0000
Iter : 8001, loss: 0.0001, accuracy: 1.0000
Iter : 9001, loss: 0.0001, accuracy: 1.0000
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.28it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9997999399819946
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.28it/s]


Evaluating reconstruction ...
Totoal accuracy : 0.9997999399819946
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.31it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.28it/s]


Evaluating reconstruction ...
Totoal accuracy : 1.0


In [53]:
reps = 10
short_term_memories = [2,4,6,8,10]
past_recalls = [1,2,3,4,5]
### initial training ###
total_samples = 10000
working_memory = 1
# short_term_memory = 10
hidden_size = 50
vocab_size = 8
embedding_dim = 5
lr = 6e-4
repitition = []
acc = []
bptt = []
recalls = []


for rep in range(reps):
    for short_term_memory in short_term_memories:
        model = RNNEncoder(vocab_size, embedding_dim, hidden_size)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-8)
        criterion = torch.nn.CrossEntropyLoss()

        data = get_sequence(total_samples, 2, 3, train_percent=1.0)

        data_set = Dataset_converter(data, working_memory, short_term_memory)
        train_loader = DataLoader(data_set, batch_size=1, shuffle=False)

        total = 0
        correct = np.zeros(1000,dtype=float)
        for X, y in train_loader:
            optimizer.zero_grad()

            if total == 0:
                y_pred, h = model(X)
            else:
                y_pred, h = model(X, hidden)

            loss = criterion(y_pred, y[0])     
            loss.backward()
            optimizer.step()

            # print(total)
            with torch.no_grad():
                hidden = h.detach()
                total += 1

                if y[0] == y_pred[0].argmax():
                        correct[total%1000] = 1
                else:
                    correct[total%1000] = 0


                test_acc = np.sum(correct)/total if total<1000 else np.sum(correct)/1000
                
                
                if total%1000 == 0:
                    print(f'Iter : {total+1}, loss: {loss:.4f}, accuracy: {test_acc:.4f}')



        ### extract the hidden states from the trained RNN ###
        hidden_states = []

        for X, y in train_loader:
            with torch.no_grad():
                if total == 0:
                    y_pred, h = model(X)
                else:
                    y_pred, h = model(X, h)

                hidden_states.append(
                    h[0][0]
                )

        data_ = []
        for ch in data:
            data_.append(ch)

        ### train classifier to reconstruct past tokens ###
        for past_recall in past_recalls:
            print('Doing recall ', past_recall)
            data_set = Dataset_reconstructer(hidden_states, data_, short_term_memory=short_term_memory, past_recall=past_recall)
            reconstruct_loader = DataLoader(data_set, batch_size=1, shuffle=False)

            reconstructor = classifier(vocab_size, hidden_size)
            optimizer = torch.optim.SGD(reconstructor.parameters(), lr=lr, momentum=0.95)
            criterion = torch.nn.CrossEntropyLoss()

            total = 0
            # correct = np.zeros(1000,dtype=float)
            print('Training reconstruction classifier ...')
            for epoch in tqdm(range(10)):
                for X, y in reconstruct_loader:
                    optimizer.zero_grad()

                    y_pred = reconstructor(X)
                    

                    loss = criterion(y_pred, y)     
                    loss.backward()
                    optimizer.step()

            print("Evaluating reconstruction ...")
            correct = 0
            for X, y in reconstruct_loader:
                with torch.no_grad():
                    y_pred = reconstructor(X)
                    
                    if y_pred.argmax() == y:
                        correct += 1

            print('Total accuracy :', correct/len(data_set))
            repitition.append(rep)
            acc.append(correct/len(data_set))
            bptt.append(short_term_memory)
            recalls.append(past_recall)
            
            
            
df = pd.DataFrame()
df['reps'] = repitition
df['accuracy'] = acc 
df['BBPTT'] = bptt 
df['Recall'] = recalls

with open('../pickle_files/memory_capacity_hard_patterned.pickle', 'wb') as f:
    pickle.dump(df, f)

Iter : 1001, loss: 1.2984, accuracy: 0.3680
Iter : 2001, loss: 0.8552, accuracy: 0.5860
Iter : 3001, loss: 2.5855, accuracy: 0.6590
Iter : 4001, loss: 1.0975, accuracy: 0.6640
Iter : 5001, loss: 0.6380, accuracy: 0.6720
Iter : 6001, loss: 1.2426, accuracy: 0.6740
Iter : 7001, loss: 1.1090, accuracy: 0.6620
Iter : 8001, loss: 0.5308, accuracy: 0.6710
Iter : 9001, loss: 1.4451, accuracy: 0.6670
Doing recall  1
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.27it/s]


Evaluating reconstruction ...
Total accuracy : 1.0
Doing recall  2
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.29it/s]


Evaluating reconstruction ...
Total accuracy : 0.9254776432929879
Doing recall  3
Training reconstruction classifier ...


...

Evaluating reconstruction ...
Total accuracy : 0.7881669836820503
Doing recall  4
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.27it/s]


Evaluating reconstruction ...
Total accuracy : 0.5956552207428171
Doing recall  5
Training reconstruction classifier ...


100%|██████████| 10/10 [00:07<00:00,  1.28it/s]


Evaluating reconstruction ...
Total accuracy : 0.46721393532886174
